# Restaurant Annual Turnover Prediction — Fourth Model Comparison

**Student register number:** submission

**Student name:** submission

**Team name:** submission

**Status:** experimental candidate; compare confirmation results before replacing the confirmed submission.

Predict annual turnover in INR for 500 test restaurants. The CSV contains
exactly `Registration Number` and `Annual Turnover` as its two columns.

**Google Colab:** upload this notebook, select a CPU runtime, and Run all.
Upload the four supplied CSV files when prompted. The final cell downloads
the prediction CSV. Save the executed notebook with its outputs using the
Colab File menu. The complete repeated validation fits many models and may
take tens of minutes on a CPU; the progress messages identify each assignment.

## 1. Objective and the results that are actually known

Annual turnover is a continuous value, so this is a **regression** problem.
The evaluation metric is `sqrt(mean((actual - prediction)**2))`, in original
INR units. Every validation restaurant is included, including extreme values.
No hidden test targets are available locally.

The supplied October 12 screenshot confirms two CSV submissions and **rank 3**.
The latest confirmed best RMSE is **12,094,600.3216516**, improved from the
first submission's **12,204,615.6438253**. The leader shown scores
**11,946,659.0702364**. This notebook generates a further candidate whose
**portal RMSE and rank are unknown until submitted**.

This round evaluated **24 settings, including the reference setting**, and a
coarse grid of **78 blends** on the existing seed-42 development predictions.
Settings and blend weights were frozen before subsequent comparisons. The
final mixture retains 25% of the submitted v3 model, adds 37.5% of a CatBoost
model using 32 numeric split borders, and 37.5% of a CatBoost model that treats
ordinal ratings as categories. Each new component averages two model seeds.

Validation assignments 1701 and 1711 were not used to select this candidate.
Assignments 71, 2026, 303 and 909 were used previously; cities form a separate
stress test. **All assignments reuse the same labeled restaurants**, whose
labels informed earlier development. These are robustness checks, not an
untouched external test or an unbiased nested model-selection estimate.
No blend is retuned using these confirmation scores or the public score.

In [ ]:
import os, sys, json, hashlib, importlib.util, subprocess, warnings
from pathlib import Path
packages={'numpy':'numpy==2.5.4','pandas':'pandas==3.0.6','sklearn':'scikit-learn==1.9.1',
          'catboost':'catboost==1.2.10','scipy':'scipy','matplotlib':'matplotlib'}
missing=[v for k,v in packages.items() if importlib.util.find_spec(k) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, catboost, scipy
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
warnings.filterwarnings('ignore',category=FutureWarning)
FILE_STEM='submission'
OUTPUT_DIR=Path.cwd()
print('Python:',sys.version.split()[0])
print('Versions:',{k:v.__version__ for k,v in {'numpy':np,'pandas':pd,'sklearn':sklearn,'catboost':catboost,'scipy':scipy}.items()})
print('Tested library versions: numpy 2.5.4, pandas 3.0.6, scikit-learn 1.9.1, catboost 1.2.10.')
print('Known first submission RMSE: 12,204,615.6438253')
print('Known latest submission RMSE: 12,094,600.3216516; screenshot rank: 3')
print('Leader shown: 11,946,659.0702364; this candidate portal score: UNKNOWN')

## 2. Load files and enforce the data contract

`Registration Number` in each prediction row is the **restaurant identifier**.
The student's register number belongs in the notebook and filenames. The
training file's `Endorsed By` spelling is normalized to the test file's spelling.

In [ ]:
required=['Train_dataset.csv','Test_dataset.csv','Data_Dictionary.csv','Sample_Solution.csv']
DATA_DIR=Path.cwd()
if not all((DATA_DIR/f).exists() for f in required):
    local=Path('~/Downloads')
    if all((local/f).exists() for f in required): DATA_DIR=local
    else:
        try: from google.colab import files
        except ImportError: raise FileNotFoundError('Place the four CSV files beside the notebook or set DATA_DIR.')
        files.upload(); DATA_DIR=Path.cwd()
assert all((DATA_DIR/f).exists() for f in required)
train=pd.read_csv(DATA_DIR/required[0]).rename(columns={'Endorsed By':'Endoresed By'})
test=pd.read_csv(DATA_DIR/required[1])
dictionary=pd.read_csv(DATA_DIR/required[2]); sample=pd.read_csv(DATA_DIR/required[3])
y=train['Annual Turnover'].to_numpy(float)
assert np.isfinite(y).all() and (y>=0).all()
assert 'Annual Turnover' not in test
assert train.drop(columns='Annual Turnover').columns.tolist()==test.columns.tolist()
assert len(test)==500 and test['Registration Number'].is_unique and train['Registration Number'].is_unique
assert set(train['Registration Number']).isdisjoint(test['Registration Number'])
assert sample.columns.tolist()==['Registration Number','Annual Turnover']
assert sample['Registration Number'].equals(test['Registration Number'])
input_hashes={f:hashlib.sha256((DATA_DIR/f).read_bytes()).hexdigest() for f in required}
print('Train:',train.shape,'Test:',test.shape)
print('PASS: schema, target validity, unique/disjoint restaurant IDs, sample headers and row order.')
print('Input SHA256 hashes:',json.dumps(input_hashes,indent=2))

## 3. Data findings and the modeling hypotheses

A small upper tail strongly affects squared error. Robust training losses and
training-only target capping are candidate regularizers; they do not change
the RMSE metric or justify removing large validation targets. Service and
comfort rating distributions differ between train and test. These observations
can motivate comparisons, but the hidden test RMSE cannot be inferred from them.

Fewer numeric split thresholds may reduce variance. Treating ordinal ratings as
categories permits unordered category groupings instead of only ordered numeric
threshold splits. Numeric trees already support nonlinear effects without an
equal-distance assumption. The retained part of the previous ensemble supplies models that
still use ratings numerically. Whether either hypothesis helps is checked below.

In [ ]:
print(pd.Series(y).describe(percentiles=[.5,.9,.95,.99]).round(0).to_string())
sq=(y-y.mean())**2; tail=y>100e6
print(f'Above INR 100m: {tail.sum()} rows; {100*sq[tail].sum()/sq.sum():.1f}% of mean-baseline squared error.')
print(pd.DataFrame({'Train':train[['Service','Comfortablility']].mean(),
                    'Test':test[['Service','Comfortablility']].mean()}).round(3).to_string())
fig,ax=plt.subplots(figsize=(8,3.5))
ax.hist(y/1e6,bins=45,color='#2563eb',edgecolor='white')
ax.set(xlabel='Annual turnover (INR millions)',ylabel='Restaurants (log scale)',yscale='log',
       title='All original training targets remain in validation')
fig.tight_layout();plt.show()

## 4. Self-contained features, models and the frozen contract

The notebook includes the complete fitting code, including the submitted model
used for comparison. Each imputer, encoder and scaler is fitted inside its
training fold. Features exclude restaurant ID and turnover. Date parsing and
categorical string normalization are deterministic; missing values receive
explicit treatment. Categorical rating values are passed directly to CatBoost.

The complete configuration below records selected models, training seeds,
weights, development results and the validation plan. Huber delta is INR40m for
the new models. Their iterations are fixed, so validation labels are not used
for early stopping. All scored labels retain their original turnover values.

In [ ]:
TARGET = 'Annual Turnover'

ID = 'Registration Number'

EVENTS = ['Live Music Rating','Comedy Gigs Rating','Value Deals Rating','Live Sports Rating']

SOCIAL = ['Facebook Popularity Quotient','Instagram Popularity Quotient']

CORE = SOCIAL + ['Hygiene Rating','Restaurant Zomato Rating','Live Music Rating','Value for Money','Overall Restaurant Rating','Restaurant City Tier','Cuisine','Restaurant Theme','Restaurant Location','Endoresed By','City','Opening Year']

RATINGS = ['Resturant Tier','Restaurant City Tier','Restaurant Zomato Rating','Order Wait Time','Staff Responsivness','Value for Money','Hygiene Rating','Food Rating','Overall Restaurant Rating','Ambience','Lively','Service','Comfortablility','Privacy'] + EVENTS

FLAGS = ['Fire Audit','Liquor License Obtained','Situated in a Multi Complex','Dedicated Parking','Open Sitting Available']

def features(raw, variant='base'):
    data=raw.drop(columns=[TARGET,ID],errors='ignore').copy()
    dates=pd.to_datetime(data.pop('Opening Day of Restaurant'),format='%d-%m-%Y',errors='coerce')
    data['Opening Year']=dates.dt.year.astype(float)
    data['Opening Month']=dates.dt.month.astype(float)
    for c in EVENTS:
        data['Offers '+c.replace(' Rating','')]=data[c].notna().astype(int)
    parts=data['Cuisine'].astype('string').fillna('Missing').str.split(',',n=1,expand=True)
    data['Cuisine 1']=parts[0].str.strip().str.lower()
    data['Cuisine 2']=parts.reindex(columns=[0,1])[1].astype('string').str.strip().str.lower()
    for c in ['Cuisine','City','Restaurant Location','Endoresed By','Restaurant Type','Restaurant Theme']:
        data[c]=data[c].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True).fillna('Missing')
    data['City Is Unknown']=(data['City']=='-1').astype(int)
    for c in data.select_dtypes(include=['object','string']).columns:
        data[c]=data[c].fillna('Missing').astype(str)
    if variant in ['engineered','engineered_drop']:
        data['City']=data['City'].replace({'bengaluru':'bangalore','gurugram':'gurgaon','bombay':'mumbai','calcutta':'kolkata','delhi':'new delhi'})
        data['Cuisine unordered']=['|'.join(sorted(v.split(','))) for v in data['Cuisine']]
        data['Opening Day']=dates.dt.day.astype(float)
        data['Opening Days Since 1970']=(dates-pd.Timestamp('1970-01-01')).dt.days.astype(float)
        data['Social mean']=data[SOCIAL].mean(axis=1)
        data['Social difference']=data[SOCIAL[0]]-data[SOCIAL[1]]
        data['Social missing count']=data[SOCIAL].isna().sum(axis=1)
        data['Rating missing count']=data.isna().sum(axis=1)
        data['Event count']=data[EVENTS].notna().sum(axis=1)
        data['Event mean']=data[EVENTS].mean(axis=1)
        data['Quality mean']=data[['Food Rating','Hygiene Rating','Overall Restaurant Rating']].mean(axis=1)
        data['Social x Zomato']=data['Social mean']*data['Restaurant Zomato Rating']
        data['Social x Hygiene']=data['Social mean']*data['Hygiene Rating']
        data['Quality x Value']=data['Quality mean']*data['Value for Money']
    if variant in ['drop_shift','engineered_drop']:
        data=data.drop(columns=['Service','Comfortablility'])
    if variant=='core': data=data[CORE]
    if variant=='ratings':
        keep=SOCIAL+['Resturant Tier','Restaurant Zomato Rating','Restaurant City Tier','Staff Responsivness','Value for Money','Hygiene Rating','Overall Restaurant Rating']+EVENTS+['Lively','Ambience','Privacy']
        data=data[keep]
    if variant=='no_city_date':
        data=data.drop(columns=['City','City Is Unknown','Opening Year','Opening Month'])
    return data

def rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))

def sklearn_model(X,spec,seed):
    cats=X.select_dtypes(include=['object','string']).columns.tolist()
    nums=[c for c in X if c not in cats]
    num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler())
    if spec['kind']=='spline':
        num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())
    pre=ColumnTransformer([('num',num,nums),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=10,sparse_output=False),cats)])
    kind=spec['kind']
    if kind in ['ridge','spline']: model=Ridge(alpha=spec.get('alpha',100))
    elif kind=='svr': model=SVR(C=spec.get('C',20),epsilon=.5,kernel='rbf')
    elif kind=='hist': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)
    elif kind=='extra': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)
    return make_pipeline(pre,model)

def fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):
    ys=np.asarray(y)[train_idx].copy()
    if 'cap' in spec: ys=np.minimum(ys,spec['cap'])
    if spec['kind']=='cat':
        params=dict(loss_function=spec.get('loss','RMSE'),eval_metric='RMSE',iterations=fixed_iterations or spec.get('iterations',1600),learning_rate=.04,depth=spec.get('depth',6),l2_leaf_reg=spec.get('l2',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)
        if 'boosting_type' in spec: params['boosting_type']=spec['boosting_type']
        if spec.get('monotone'): params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
        model=CatBoostRegressor(**params)
        cat=X.select_dtypes(include=['object','string']).columns.tolist()
        stopping={} if fixed_iterations or 'iterations' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)
        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)
        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_
    elif spec['kind'] in ['lgb','xgb']:
        pre=sklearn_model(X,dict(kind='ridge'),seed)[:-1]
        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])
        early=not (fixed_iterations or 'iterations' in spec)
        iterations=fixed_iterations or spec.get('iterations',1400)
        if spec['kind']=='lgb':
            from lightgbm import LGBMRegressor,early_stopping
            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get('leaves',15),min_child_samples=spec.get('leaf_samples',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get('objective','regression'))
            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric='rmse',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}
            model.fit(ztr,ys/1e6,**params)
            trees=model.best_iteration_ if early else iterations
        else:
            from xgboost import XGBRegressor
            params=dict(n_estimators=iterations,max_depth=spec.get('depth',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method='hist',objective=spec.get('objective','reg:squarederror'),eval_metric='rmse',base_score=float(ys.mean()/1e6))
            if early: params['early_stopping_rounds']=100
            if spec.get('objective')=='reg:pseudohubererror': params['huber_slope']=12
            model=XGBRegressor(**params)
            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))
            trees=model.best_iteration+1 if early else iterations
        p=model.predict(zva)*1e6
    else:
        model=sklearn_model(X,spec,seed)
        model.fit(X.iloc[train_idx],ys/1e6)
        p=model.predict(X.iloc[valid_idx])*1e6; trees=0
    if return_fitted:
        fitted=dict(model=model,preprocessor=pre if spec['kind'] in ['lgb','xgb'] else None,scale=1 if spec['kind']=='cat' else 1e6)
        return np.maximum(p,0),trees,fitted
    return np.maximum(p,0),trees

def predict_fitted(fitted,X):
    transformed=fitted['preprocessor'].transform(X) if fitted['preprocessor'] is not None else X
    return np.maximum(fitted['model'].predict(transformed)*fitted['scale'],0)

def cat_fit(X, y, spec, seed):
    params=dict(iterations=spec['iterations'], learning_rate=.04, depth=spec['depth'],
                l2_leaf_reg=spec.get('l2',5), random_seed=seed, thread_count=2,
                verbose=False, allow_writing_files=False,
                loss_function=f"Huber:delta={spec['delta']*1e6:g}" if 'delta' in spec else 'RMSE')
    if spec.get('monotone'):
        params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
    if spec.get('ordered'): params['boosting_type']='Ordered'
    if 'subsample' in spec: params.update(bootstrap_type='Bernoulli',subsample=spec['subsample'])
    if 'random_strength' in spec: params['random_strength']=spec['random_strength']
    fitted=CatBoostRegressor(**params)
    target=np.minimum(y,spec['cap']*1e6) if 'cap' in spec else y
    fitted.fit(X,target,cat_features=X.select_dtypes(include=['object','string']).columns.tolist())
    return fitted

def feature_frame(raw, variant='base'):
    data=raw.rename(columns={'Endorsed By':'Endoresed By'}).drop(columns=['Annual Turnover','Registration Number'],errors='ignore').copy()
    dates=pd.to_datetime(data.pop('Opening Day of Restaurant'),format='%d-%m-%Y',errors='coerce')
    data['Opening Year']=dates.dt.year.astype(float)
    data['Opening Month']=dates.dt.month.astype(float)
    for c in EVENTS:
        data['Offers '+c.replace(' Rating','')]=data[c].notna().astype(int)
    parts=data['Cuisine'].astype('string').fillna('Missing').str.split(',',n=1,expand=True)
    data['Cuisine 1']=parts[0].str.strip().str.lower()
    data['Cuisine 2']=parts.reindex(columns=[0,1])[1].astype('string').str.strip().str.lower()
    for c in ['Cuisine','City','Restaurant Location','Endoresed By','Restaurant Type','Restaurant Theme']:
        data[c]=data[c].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True).fillna('Missing')
    data['City Is Unknown']=(data['City']=='-1').astype(int)
    for c in data.select_dtypes(include=['object','string']).columns:
        data[c]=data[c].fillna('Missing').astype(str)
    if variant == 'categorical_ratings':
        for c in RATINGS:
            data[c]=data[c].map(lambda v: 'Missing' if pd.isna(v) else f'{v:g}')
    elif variant == 'dual_ratings':
        for c in RATINGS:
            data[c+' Category']=data[c].map(lambda v: 'Missing' if pd.isna(v) else f'{v:g}')
    elif variant == 'categorical_tiers_flags':
        for c in ['Resturant Tier','Restaurant City Tier']+FLAGS:
            data[c]=data[c].map(lambda v: 'Missing' if pd.isna(v) else f'{v:g}')
    elif variant == 'social_quantized':
        for c in SOCIAL:
            data[c]=np.round(data[c]/5)*5
    elif variant == 'events_zero':
        data[EVENTS]=data[EVENTS].fillna(0)
        data['Event Count']=data[['Offers '+c.replace(' Rating','') for c in EVENTS]].sum(axis=1)
    elif variant == 'date_detail':
        data['Opening Days Since 1970']=(dates-pd.Timestamp('1970-01-01')).dt.days.astype(float)
        data['Opening Day of Month']=dates.dt.day.astype(float)
        data['Opening Day of Week']=dates.dt.dayofweek.astype(float)
        data['Opening Month Sin']=np.sin(2*np.pi*dates.dt.month/12)
        data['Opening Month Cos']=np.cos(2*np.pi*dates.dt.month/12)
    elif variant == 'city_cuisine':
        data['City']=data['City'].replace({'bengaluru':'bangalore','gurugram':'gurgaon','bombay':'mumbai','calcutta':'kolkata','delhi':'new delhi'})
        data['City Cuisine']=data['City']+'|'+data['Cuisine']
        data['Location Cuisine']=data['Restaurant Location']+'|'+data['Cuisine']
        data['Tier Cuisine']=data['Resturant Tier'].fillna(-1).astype(str)+'|'+data['Cuisine']
    elif variant == 'reduced':
        # A fixed hypothesis excluding subjective service/comfort scores and sparse event fields.
        data=data.drop(columns=['Service','Comfortablility','Comedy Gigs Rating','Value Deals Rating','Live Sports Rating'])
    elif variant == 'cuisine_canonical':
        data['Cuisine']=['|'.join(sorted(part.strip() for part in value.split(','))) for value in data['Cuisine']]
        data=data.drop(columns=['Cuisine 1','Cuisine 2'])
    elif variant != 'base':
        raise ValueError(variant)
    return data

def get_splits(train,scheme):
    if str(scheme)=='city':
        group=features(train,'engineered')['City']
        splits=list(GroupKFold(5).split(train,groups=group))
        for tr,va in splits:assert set(group.iloc[tr]).isdisjoint(group.iloc[va])
    else:splits=list(KFold(5,shuffle=True,random_state=int(scheme)).split(train))
    counts=np.zeros(len(train),int)
    for tr,va in splits:
        assert set(tr).isdisjoint(va);counts[va]+=1
    assert (counts==1).all()
    return splits

def new_fit(X,y,component,seed):
    model=CatBoostRegressor(**component['parameters'],random_seed=seed,thread_count=2,verbose=False,allow_writing_files=False)
    weights=None
    if 'support_columns' in component:
        weights=np.where(X[component['support_columns']].isna().any(axis=1),component['missing_weight'],1.)
    model.fit(X,np.asarray(y),cat_features=X.select_dtypes(include=['object','string']).columns.tolist(),sample_weight=weights)
    return model

def compute_v3(train,y,scheme,contract):
    splits=get_splits(train,scheme);pred=np.zeros(len(y))
    for name,spec in contract['v2_configs'].items():
        if name not in contract['v2_weights']:continue
        X=features(train,spec['variant']);component=np.zeros(len(y))
        for f,(tr,va) in enumerate(splits):
            component[va],_=fit_predict(X,y,tr,va,spec,43+f,spec.get('iterations'))
        pred+=.5*contract['v2_weights'][name]*component
    X=features(train,'base');spec=contract['v3_parameters']
    for f,(tr,va) in enumerate(splits):
        parts=[]
        for seed in contract['v3_seeds']:
            model=cat_fit(X.iloc[tr],y[tr],spec,seed+f)
            parts.append(np.maximum(model.predict(X.iloc[va]),0))
        pred[va]+=.5*np.mean(parts,axis=0)
    return pred

def new_oof(train,y,scheme,component,seeds):
    X=feature_frame(train,component['variant']);individual=np.zeros((len(y),len(seeds)))
    for f,(tr,va) in enumerate(get_splits(train,scheme)):
        for j,seed in enumerate(seeds):
            model=new_fit(X.iloc[tr],y[tr],component,seed+f)
            individual[va,j]=np.maximum(model.predict(X.iloc[va]),0)
    return individual.mean(axis=1)

def full_fit(train,test,y,contract):
    fitted_v2={};fitted_v3=[];fitted_new={}
    previous=np.zeros(len(test));idx=np.arange(len(train));vidx=np.arange(len(train),len(train)+len(test))
    combined_y=np.r_[y,np.full(len(test),np.nan)]
    for name,w in contract['v2_weights'].items():
        spec=contract['v2_configs'][name]
        X=pd.concat([features(train,spec['variant']),features(test,spec['variant'])],ignore_index=True)
        p,_,fitted=fit_predict(X,combined_y,idx,vidx,spec,42,spec.get('iterations'),return_fitted=True)
        fitted_v2[name]=fitted;previous+=.5*w*p
    X=features(train,'base');T=features(test,'base')
    for seed in contract['v3_seeds']:
        model=cat_fit(X,y,contract['v3_parameters'],seed);fitted_v3.append(model)
        previous+=.5*np.maximum(model.predict(T),0)/len(contract['v3_seeds'])
    pred=contract['v3_weight']*previous
    for name,component in contract['components'].items():
        X=feature_frame(train,component['variant']);T=feature_frame(test,component['variant']);fitted_new[name]=[]
        for seed in contract['seeds']:
            model=new_fit(X,y,component,seed);fitted_new[name].append(model)
            pred+=component['weight']*np.maximum(model.predict(T),0)/len(contract['seeds'])
    return previous,pred,{'v2':fitted_v2,'v3':fitted_v3,'new':fitted_new}

def predict_final(raw,fitted,contract):
    old=sum(.5*w*predict_fitted(fitted['v2'][name],features(raw,contract['v2_configs'][name]['variant'])) for name,w in contract['v2_weights'].items())
    X=features(raw,'base')
    old+=.5*np.mean([np.maximum(model.predict(X),0) for model in fitted['v3']],axis=0)
    p=contract['v3_weight']*old
    for name,component in contract['components'].items():
        X=feature_frame(raw,component['variant'])
        p+=component['weight']*np.mean([np.maximum(model.predict(X),0) for model in fitted['new'][name]],axis=0)
    return p

def check_submission(frame,test):
    assert frame.shape==(500,2),'Expected 500 rows and two columns'
    assert frame.columns.tolist()==['Registration Number','Annual Turnover'],'Incorrect header'
    assert frame['Registration Number'].equals(test['Registration Number']),'Restaurant IDs/order changed'
    assert frame['Registration Number'].is_unique,'Duplicate restaurant IDs'
    assert np.isfinite(frame['Annual Turnover']).all(),'Non-finite prediction'
    assert (frame['Annual Turnover']>=0).all(),'Negative turnover'

In [ ]:
CONTRACT=json.loads('{"version": "v4", "v2_configs": {"engineered": {"kind": "cat", "variant": "engineered", "depth": 6, "iterations": 260}, "engineered_drop": {"kind": "cat", "variant": "engineered_drop", "depth": 6, "iterations": 216}, "ordered_depth5": {"kind": "cat", "variant": "base", "depth": 5, "boosting_type": "Ordered", "iterations": 219}, "huber20_engineered": {"kind": "cat", "variant": "engineered_drop", "depth": 5, "loss": "Huber:delta=20000000", "iterations": 699}, "huber40": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 664}, "clip120": {"kind": "cat", "variant": "base", "depth": 5, "cap": 120000000.0, "iterations": 591}, "spline_ridge": {"kind": "spline", "variant": "base", "alpha": 100}, "svr_core": {"kind": "svr", "variant": "core", "C": 20}, "huber40_engineered": {"kind": "cat", "variant": "engineered", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 619}, "huber40_monotone": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "monotone": true, "iterations": 447}}, "v2_weights": {"engineered": 0.06448907249408362, "engineered_drop": 0.09631834163153197, "ordered_depth5": 0.08507039177778712, "huber20_engineered": 0.06274452861282048, "huber40": 0.08855362607985372, "clip120": 0.07794631558814363, "spline_ridge": 0.15648185431084338, "svr_core": 0.058428294374292714, "huber40_engineered": 0.09327747966602483, "huber40_monotone": 0.2166900954646185}, "v3_parameters": {"variant": "base", "depth": 5, "iterations": 850, "delta": 40, "subsample": 0.7, "l2": 10}, "v3_seeds": [1042, 2042, 3042], "v3_weight": 0.25, "seeds": [1042, 2042], "components": {"parameter_borders32": {"variant": "base", "parameters": {"iterations": 850, "learning_rate": 0.04, "depth": 5, "l2_leaf_reg": 10, "loss_function": "Huber:delta=40000000", "bootstrap_type": "Bernoulli", "subsample": 0.7, "border_count": 32}, "weight": 0.375}, "feature_categorical_ratings": {"variant": "categorical_ratings", "parameters": {"iterations": 850, "learning_rate": 0.04, "depth": 5, "l2_leaf_reg": 10, "loss_function": "Huber:delta=40000000", "bootstrap_type": "Bernoulli", "subsample": 0.7}, "weight": 0.375}}, "development": {"baseline_rmse": 19839027.540841296, "candidate_rmse": 19823848.416554824, "settings_evaluated": 24, "blends_evaluated": 78, "selection_split": 42}, "validation_plan": {"new_splits": [1701, 1711], "previous_splits": [71, 2026, 303, 909], "stress": ["city"], "warning": "Resplits of reused training labels, not an external test; no retuning on these checks."}}')

assert np.isclose(sum(CONTRACT['v2_weights'].values()),1)
assert np.isclose(CONTRACT['v3_weight']+sum(s['weight'] for s in CONTRACT['components'].values()),1)
assert np.isclose(rmse([0,3],[0,0]),np.sqrt(4.5))
changed=train.copy();changed['Annual Turnover']=0;changed['Registration Number']=-1
variants={s['variant'] for s in CONTRACT['v2_configs'].values()}
for variant in variants:
    X=features(train,variant)
    assert X.columns.equals(features(test,variant).columns)
    assert 'Annual Turnover' not in X and 'Registration Number' not in X
    pd.testing.assert_frame_equal(X,features(changed,variant))
for component in CONTRACT['components'].values():
    variant=component['variant'];X=feature_frame(train,variant)
    assert X.columns.equals(feature_frame(test,variant).columns)
    assert 'Annual Turnover' not in X and 'Registration Number' not in X
    pd.testing.assert_frame_equal(X,feature_frame(changed,variant))
print('PASS: toy RMSE formula, weight sums, target/ID exclusion and feature schema alignment.')
print('Frozen development comparison:',json.dumps(CONTRACT['development'],indent=2))
print('Previous model weight:',CONTRACT['v3_weight'])
print('New components:',json.dumps(CONTRACT['components'],indent=2))
ctr=feature_frame(train,'categorical_ratings');cte=feature_frame(test,'categorical_ratings')
coverage=[]
for c in RATINGS:
    coverage.append({'Feature':c,'Train categories':ctr[c].nunique(),
                     'Unseen test rows':int((~cte[c].isin(set(ctr[c]))).sum())})
print('Categorical coverage:',pd.DataFrame(coverage).to_string(index=False),sep='\n')

## 5. Compare frozen models on identical validation assignments

Every restaurant receives exactly one held-out prediction per five-fold
assignment. Known canonical city aliases are combined for the separate city
stress check. The earlier city-normalization rules are retained so that the
old and new models use precisely the same comparison groups; city spellings
outside that explicit map remain a grouping limitation.

The local executed notebook may reuse **actual model predictions** via the
explicit `MLA_VALIDATION_CACHE` environment variable. Input hashes, the complete
configuration hash, and prediction-file hashes must match. The cell prints
each verified reuse. With that variable absent, including normal Colab use,
`compute_validation(..., use_prior_cache=False)` trains every fold below.

In [ ]:
def compute_validation(train,y,scheme,contract,use_prior_cache=False):
    if use_prior_cache:
        raise ValueError('Use the explicit hash-verified cache loader below, or train with use_prior_cache=False.')
    previous=compute_v3(train,y,scheme,contract)
    prediction=contract['v3_weight']*previous
    for name,component in contract['components'].items():
        prediction+=component['weight']*new_oof(train,y,scheme,component,contract['seeds'])
    return {'previous':previous,'candidate':prediction}

cache_value=os.environ.get('MLA_VALIDATION_CACHE')
cache_root=Path(cache_value) if cache_value else None
manifest=None
if cache_root is not None:
    manifest=json.loads((cache_root/'cache_manifest.json').read_text())
    assert manifest['input_hashes']==input_hashes,'Cached input files changed'
    digest=hashlib.sha256(json.dumps(CONTRACT,sort_keys=True).encode()).hexdigest()
    assert manifest['configuration_sha256']==digest,'Cached configuration changed'
validation_predictions={};rows=[]
schemes=[1701,1711,71,2026,303,909,'city']
for scheme in schemes:
    get_splits(train,scheme) # Fold/group coverage checks run even with a verified cache.
    filename=f'comparison_{scheme}.npz'
    if cache_root is not None:
        assert hashlib.sha256((cache_root/filename).read_bytes()).hexdigest()==manifest['files'][filename]
        cached=np.load(cache_root/filename)
        predictions={k:cached[k] for k in ['previous','candidate']}
        print('Verified executed-fold cache:',filename)
    else:
        print('Training complete validation:',scheme,flush=True)
        predictions=compute_validation(train,y,scheme,CONTRACT,use_prior_cache=False)
    assert all(p.shape==y.shape and np.isfinite(p).all() and (p>=0).all() for p in predictions.values())
    validation_predictions[str(scheme)]=predictions
    rows.append({'Validation':str(scheme),'Type':'City stress' if scheme=='city' else 'New split' if scheme in [1701,1711] else 'Previous split',
                 'Previous submitted RMSE':rmse(y,predictions['previous']),
                 'Current candidate RMSE':rmse(y,predictions['candidate'])})
validation_table=pd.DataFrame(rows)
validation_table['Gain vs previous submitted']=validation_table['Previous submitted RMSE']-validation_table['Current candidate RMSE']
print(validation_table.to_string(index=False,float_format=lambda x:f'{x:,.2f}'))
metric_columns=['Previous submitted RMSE','Current candidate RMSE','Gain vs previous submitted']
random_table=validation_table[validation_table['Validation']!='city']
means=random_table[metric_columns].mean()
fresh_means=validation_table[validation_table['Type']=='New split'][metric_columns].mean()
print('Mean across all six shuffled assignments:')
print(means.to_string(float_format=lambda x:f'{x:,.2f}'))
print('Mean of two split assignments reserved during this round selection:')
print(fresh_means.to_string(float_format=lambda x:f'{x:,.2f}'))
promote_candidate=(fresh_means['Gain vs previous submitted']>0 and means['Gain vs previous submitted']>0)
print('Decision:', 'Candidate passes the positive-mean comparison rule; hidden-test performance remains unknown.'
      if promote_candidate else 'Retain the confirmed v3 submission: this experimental candidate does not pass both positive-mean comparisons.')

## 6. Inferences, error concentration and uncertainty

Confirmation results are shown with their actual signs: a negative reduction
means the new candidate worsened. The city stress result is reported separately
and is excluded from random-split averages. Repeated assignments are correlated
because they reuse restaurants. They cannot be counted as independent datasets.

The paired bootstrap resamples restaurants' already-computed squared errors,
averaging their errors across assignments. Its interval is conditional on
these fitted and selected models. It does not capture full retraining,
model-selection optimism, or differences in the hidden test distribution.
Target-tail diagnostics do not remove any rows from the primary metric.

In [ ]:
shuffled=[1701,1711,71,2026,303,909]
previous_sq=np.mean([(y-validation_predictions[str(s)]['previous'])**2 for s in shuffled],axis=0)
candidate_sq=np.mean([(y-validation_predictions[str(s)]['candidate'])**2 for s in shuffled],axis=0)
rng=np.random.default_rng(812);gains=[]
for _ in range(2000):
    ix=rng.integers(0,len(y),len(y))
    gains.append(np.sqrt(previous_sq[ix].mean())-np.sqrt(candidate_sq[ix].mean()))
gain_ci=np.quantile(gains,[.025,.975])
print('Conditional paired 95% interval for RMSE reduction:',np.round(gain_ci,2).tolist())
if gain_ci[0]<=0<=gain_ci[1]:
    print('Inference: the interval includes zero; the incremental gain is uncertain under this check.')
elif gain_ci[0]>0:
    print('Inference: this conditional interval favors the candidate; it does not establish a hidden-test improvement.')
else:
    print('Inference: this conditional interval favors the submitted model.')
wins=int((random_table['Gain vs previous submitted']>0).sum())
print(f'Shuffled validation comparisons improved: {wins} of {len(random_table)}.')
city_gain=float(validation_table.loc[validation_table['Validation']=='city','Gain vs previous submitted'].iloc[0])
print(f'City stress test reduction: {city_gain:,.2f}; '+('improved' if city_gain>0 else 'worsened or tied'))
for label,mask in [('At most INR 100m',y<=100e6),('Above INR 100m',y>100e6)]:
    print(label,': rows=',mask.sum(),'; previous RMSE=',round(float(np.sqrt(previous_sq[mask].mean())),2),
          '; candidate RMSE=',round(float(np.sqrt(candidate_sq[mask].mean())),2),
          '; candidate squared-error share=',round(float(100*candidate_sq[mask].sum()/candidate_sq.sum()),2),'%')
fig,axes=plt.subplots(1,2,figsize=(13,4.8))
x=np.arange(len(validation_table));labels=['1701*','1711*','71','2026','303','909','City']
for column,color in [('Previous submitted RMSE','#2563eb'),('Current candidate RMSE','#0f766e')]:
    axes[0].plot(x[:6],validation_table[column].iloc[:6]/1e6,'o-',label=column.replace(' RMSE',''),color=color)
    axes[0].scatter(x[-1],validation_table[column].iloc[-1]/1e6,marker='D',color=color,s=55)
axes[0].set_xticks(x,labels)
axes[0].set(xlabel='*New split assignment; City is a separate stress test',ylabel='RMSE (INR millions)',title='Validation on all original turnover values')
axes[0].legend(fontsize=8);axes[0].grid(axis='y',alpha=.2)
g=validation_table['Gain vs previous submitted'].to_numpy()/1000
axes[1].bar(x,g,color=['#0f766e' if v>=0 else '#b45309' for v in g])
axes[1].axhline(0,color='#334155',linewidth=1)
axes[1].set_xticks(x,labels)
axes[1].set(xlabel='*New split assignment; City excluded from means',ylabel='RMSE reduction (INR thousands)',title='Positive means improved; negative means worse')
for ax in axes:
    ax.axvline(5.5,color='#64748b',linestyle='--',linewidth=1)
    ax.axvspan(5.55,6.45,color='#f1f5f9',alpha=.8,zorder=-1)
fig.tight_layout();plt.show()

## 7. Fit the frozen models on all labeled rows

This cell refits every selected model on all training restaurants. It does not
substitute cached final predictions. The optional local cache contains a
separately computed full fit; its hash is verified and the freshly generated
predictions must match it. In Colab, the local reference file is unnecessary.

In [ ]:
previous_test_prediction,test_prediction,fitted=full_fit(train,test,y,CONTRACT)
assert np.allclose(predict_final(test,fitted,CONTRACT),test_prediction,rtol=1e-12,atol=.01)
if cache_root is not None:
    filename='full_prediction_parts.npz'
    assert hashlib.sha256((cache_root/filename).read_bytes()).hexdigest()==manifest['files'][filename]
    independent=np.load(cache_root/filename)
    assert np.allclose(independent['previous'],previous_test_prediction,rtol=1e-12,atol=.01)
    assert np.allclose(independent['candidate'],test_prediction,rtol=1e-12,atol=.01)
    print('PASS: freshly trained final predictions reproduce the independent full fit within INR 0.01.')
print('Mean absolute change from the submitted CSV (INR):',float(np.abs(test_prediction-previous_test_prediction).mean()))
print('Candidate range:',float(test_prediction.min()),'to',float(test_prediction.max()))

## 8. Behavior tests and exact submission file format

File checks establish that all 500 test restaurants receive valid predictions.
They do not establish prediction accuracy without the hidden target values.
Deliberately invalid files are rejected to exercise the validation rules.

In [ ]:
perm=np.random.default_rng(91).permutation(len(test))
assert np.allclose(predict_final(test.iloc[perm],fitted,CONTRACT),test_prediction[perm],rtol=1e-10,atol=.01)
edge=test.iloc[:2].copy();edge['City']='unseen_validation_city';edge['Cuisine']=pd.NA
edge['Facebook Popularity Quotient']=np.nan;edge['Live Music Rating']=np.nan
edge['Hygiene Rating']=99 # Unseen category in the ordinal representation.
edge_prediction=predict_final(edge,fitted,CONTRACT)
assert np.isfinite(edge_prediction).all() and (edge_prediction>=0).all()
submission=pd.DataFrame({'Registration Number':test['Registration Number'],'Annual Turnover':test_prediction})
check_submission(submission,test)
bad_cases=[submission.iloc[:-1],submission.iloc[::-1].reset_index(drop=True),
           submission.rename(columns={'Annual Turnover':'Turnover'}),submission.assign(index=0)]
for invalid in [np.nan,np.inf,-1]:
    bad=submission.copy();bad.loc[0,'Annual Turnover']=invalid;bad_cases.append(bad)
duplicate=submission.copy();duplicate.loc[1,'Registration Number']=duplicate.loc[0,'Registration Number'];bad_cases.append(duplicate)
for bad in bad_cases:
    rejected=False
    try:check_submission(bad,test)
    except AssertionError:rejected=True
    assert rejected,'Invalid submission was not rejected'
output_path=OUTPUT_DIR/(FILE_STEM+'.csv')
submission.to_csv(output_path,index=False)
roundtrip=pd.read_csv(output_path);check_submission(roundtrip,test)
assert np.allclose(roundtrip['Annual Turnover'],test_prediction,rtol=1e-12,atol=.001)
assert len(output_path.read_text().splitlines())==501
print('PASS: row permutation, unseen/missing features, eight deliberately invalid files, CSV schema and round-trip.')
print('Exactly 500 rows plus one header; columns:',submission.columns.tolist())
print('Saved:',output_path.name)
print('SHA256:',hashlib.sha256(output_path.read_bytes()).hexdigest())
print(submission.head().to_string(index=False))

## 9. Conclusions and submission

**Measured outcome:** the candidate worsened both split assignments reserved during this round, with an average RMSE increase of **4,388.77**. The city stress test worsened by **19,864.08**. Its six-shuffle average improved by **6,176.11**, but the conditional interval includes zero. Retain the confirmed v3 submission as the preferred model; v4 is an experimental alternative.

- The submitted v3 model's portal RMSE is **12,094,600.3216516**. That is the
  observed result for the previous file; it must not be assigned to this new CSV.
- The new candidate was selected using local development RMSE. The output above
  reports the fresh-split checks, previously used splits, city stress result and
  conditional uncertainty. Mixed results must be retained in the conclusion.
- Fewer numeric thresholds and categorical ratings are plausible variance
  controls. The tests show whether their benefit survives new fold assignments;
  they do not prove a benefit on a different restaurant population.
- Training and validation share the same labeled source dataset across repeated
  experiments. No hidden labels, competitor predictions or test-score-derived
  prediction adjustments are used. Further search can overfit development data.
- To beat the leader in the latest supplied screenshot, a new submission must
  achieve **below 11,946,659.0702364**, assuming the leader's score is unchanged.
  This candidate's portal score and rank remain unknown. Local gains cannot be
  subtracted from the prior portal RMSE to predict its next result.

Retain the confirmed v3 submission if the comparisons above do not support a
replacement. If you choose to evaluate this experimental candidate, upload its
**CSV** to **assignment M.Tech. MLA Hackathon -Restaurant Turnover Prediction**, then
retain the returned score with that file's SHA256. Upload the
final executed **notebook** to **LLJ_Oct_2026** when model selection is complete;
the supplied assignment instructions permit only one notebook submission.
Review the code and inferences so you can explain the modeling choices.

In [ ]:
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print('Prediction CSV is ready in the notebook folder.')